# A PINNs approach for the computation of eigenvalues in elliptic problems — examples

**J. Fernández Bonder & A. Salort** ([arXiv:2507.03126](https://arxiv.org/abs/2507.03126))

This notebook walks through every example of the current version of the paper (`paper/v4/FBS-v4.tex`) using the
`examples` package of this repository:

* `examples/configs.py` — one `Example` per figure/table, with the settings stated in the paper
  (Table *Settings of the linear examples* and Sections 3–5);
* `examples/core.py` — the driver: training along the E-grid, detection of the minima, closed-form refinement and
  a posteriori quantities (Section 3.1), JSON output and figures;
* `pinn_eigen.py` — the method itself (network, loss, training, samplers, boundary factors, potentials,
  `a_posteriori`).

The same examples can be run from a terminal: `python -m examples.run --list`, `python -m examples.run disk --quick`.

**Quick mode.** With `PAPER_SETTINGS = False` (default) every example runs with tiny settings
(a 3×32 network, 100 epochs per node, 500 points, at most 7 nodes per scan and 2 windows): each cell
takes a few seconds and the numbers are **not** accurate — the point is to see the pipeline. Set
`PAPER_SETTINGS = True` to use the settings of the paper (from minutes to hours per example on a CPU; the
cost of every example is discussed in the paper and in `experiments/`).

In [ ]:
import os, sys, json
from dataclasses import replace
from pathlib import Path
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pinn_eigen as pe
from examples.configs import EXAMPLES, Example, Domain, grid, linspace, three_nodes
from examples.core import run_example
from examples import references as R

PAPER_SETTINGS = False          # True: settings of the paper (slow)
SEED = 0
OUT = ROOT / "examples_out"     # JSON + PNG files are written here
pe.set_threads(2)

def run(name, show_figures=True, **overrides):
    # run one example (quick or paper settings) and show its figures
    ex = EXAMPLES[name]
    if not PAPER_SETTINGS:
        ex = ex.quick()
    if overrides:
        ex = replace(ex, **overrides)
    res = run_example(ex, seed=SEED, out_dir=OUT, verbose=False)
    from examples.core import print_summary
    print_summary(ex, res)
    if show_figures:
        for f in res.get("figures", []):
            display(Image(filename=f, width=720))
    return res

In [ ]:
from examples.run import list_examples
list_examples()

## Section 3 — the method, the unit disk

For each node $E^i$ of a grid the network $u=B(x)\,\mathrm{NN}(x)$ (here $B=1-|x|^2$) is trained to minimize
$$\bar\Lambda(u,E)=\int_\Omega(\Delta u-Vu+Eu)^2\,dx+\mu\Big(\int_\Omega u^2\,dx-1\Big)^2$$
(Adam, lr $10^{-3}$, fresh Monte Carlo sample of $N=5000$ points per epoch, early stopping 300/$10^{-5}$,
warm start from the previous node). The *recorded* curve $\mathcal R_\mu(E^i)$ is the residual term at the trained
network; its local minima are the candidate eigenvalues. Each one is refined, with the network frozen, by the
closed-form quotient
$$\hat E=\rho(u)=-\frac{\int u(\Delta u-Vu)}{\int u^2},$$
and certified by the Weinstein enclosure $[\rho-\eta,\rho+\eta]$, $\eta=\|(L-\rho)u\|/\|u\|$, which contains an
eigenvalue whatever the quality of the training. If a gap $\gamma$ to the rest of the spectrum is known, the
Kato–Temple bound $|\hat E-E_k|\le\eta^2/\gamma$ applies (here the gap is taken from the exact spectrum, so it is
a check, not a certificate). The plots show $\mathcal R_\mu$, the normalized loss $r^2$ on fresh points,
the exact curve $\ell(E)^2=\min_k(E-E_k)^2$ and the curve $\ell^2(1-\ell^2/2\mu)$ of Proposition 3.1.

**Paper (Fig. lossNN):** window $[0,32]$, $\mu=100$, at most 3000 epochs, warm start.

In [ ]:
res = run("disk")

## Section 4.1 — higher eigenvalues: the unit square

* `square_44_55` (Fig. eigen55): window $[44,55]$, $\mu=3000$; one minimum at the double eigenvalue $5\pi^2$.
* `square` (Table 1): window $[0,105]$, 80 nodes, $\mu=500$, at most 5000 epochs, re-initialization every 25 nodes.
  The paper reports $19.739909$, $49.348167$, $78.956161$, $98.695973$ (exact $2\pi^2,5\pi^2,8\pi^2,10\pi^2$), with
  $r=0.27$–$0.44$: errors two to four orders of magnitude below $r$, since the error of the quotient is quadratic in
  the residual. Double eigenvalues appear as a single minimum.

In [ ]:
res = run("square_44_55")

In [ ]:
res = run("square")

## Section 4.2 — higher dimensions: balls of $\mathbb R^3$, $\mathbb R^4$, $\mathbb R^{10}$

Exact eigenvalues $j_{m+d/2-1,n}^2$ (`examples/references.py`).

* `ball3` (Fig. loss3D left): $[0,70]$, $\mu=100$, re-initialization every 15 nodes.
* `ball3_windows`, `ball4_windows` (Table 2): three nodes $E_k-0.9,E_k+0.1,E_k+1.1$ per eigenvalue, fresh network at
  the first; the value is the quotient at the node of smallest normalized loss. Paper: $9.8757, 20.1912, 33.2178,
  39.4789$ ($d=3$) and $14.6851$, $26.3751$ ($d=4$), Temple bracket $14.666\le E_1\le14.685$.
* `ball4` (Fig. loss3D right): $E=0.7,2.2,\dots,54.7$, $\mu=200$, warm start; exactly four local minima, quotients
  $14.6828$, $26.3738$, $40.7083$, $49.2208$.
* `ball10`: $E=54,57,\dots,84$, $\mu=2000$ (a cold start at large $E$ with $\mu\ll E^2/2$ collapses to $u=0$);
  two wells, quotients within $0.1$–$1\%$ of $57.583$ and $76.939$ (multiplicity 10).

In [ ]:
res = run("ball3")

In [ ]:
res = run("ball3_windows")

In [ ]:
res = run("ball4")

In [ ]:
res = run("ball4_windows")

In [ ]:
res = run("ball10")

## Section 4.3 — other geometries: triangle and annulus

The boundary factor is $B=x_1x_2(1-x_1-x_2)$ for the triangle and $(|x|-\tfrac12)(1-|x|)$ for the annulus.

* `triangle` (Fig. loss.lap.triang): $[0,180]$, $\mu=15000$, $N=4000$, re-initialization every 50 nodes.
* `triangle_windows` (Table 4): paper $49.3509$, $98.7067$, $128.4612$, $167.7923$ (exact $5\pi^2,10\pi^2,13\pi^2,17\pi^2$);
  the cold first node of every window did not converge, the values come from the second or third node.
* `annulus` (Fig. loss.lap.annulus): the paper's settings were not recorded; the configuration here is our
  choice. In the paper the warm-started network stayed on the first eigenvalue $39.0133$ and missed $40.8725$ and
  $46.4285$ (an example of missed eigenvalues).

In [ ]:
res = run("triangle")

In [ ]:
res = run("triangle_windows")

In [ ]:
res = run("annulus")

## Section 4.4 — other potentials

Harmonic potential $V=\frac{\omega^2}{2}|x|^2$ in the unit disk ($\omega=1$, $10$; $\mu=100$), double Gaussian well
($\mu=400$) and four compactly supported wells ($\mu=1000$, $N=2000$) in the unit ball of $\mathbb R^3$. References
are finite element values (`experiments/fem`). For $\omega=10$ the trivial state $u=0$ is the minimizer at $E=0$
($\ell(0)^2=204>2\mu$), and $45.09$/$46.29$ appear as a single minimum followed by a spurious one near $49.4$.

In [ ]:
res = run("harmonic_w1")

In [ ]:
res = run("harmonic_w10")

In [ ]:
res = run("double_well")

In [ ]:
res = run("four_well")

## Section 4.5 — validation with finite elements

Seven-node windows (step 0.5) centred at the finite element eigenvalues (an accuracy test, not a detection), and a
blind scan of the double well on $[4.7,49.7]$ (step 1.5). The well potentials have clusters of close eigenvalues:
the loss curve has one minimum per cluster and the quotient returns a value inside it. Paper (seed 0): double well
$9.5783, 19.7640, 32.9552, 39.1944$; four well $9.8593, 20.1505, 33.2141, 39.4578$; harmonic $\omega=10$
$14.2899, 29.1357, 45.0892, 62.5792$; blind scan: five minima with quotients $9.5779, 19.7545, 32.7966, 39.1945,
48.5270$.

In [ ]:
res = run("double_well_windows")

In [ ]:
res = run("four_well_windows")

In [ ]:
res = run("harmonic_w10_windows")

In [ ]:
res = run("double_well_blind")

## Section 5 — the p-Laplacian

Homogeneous loss $\bar\Lambda_p(u,E)=\int(\Delta_pu+E|u|^{p-2}u)^2+\mu(\int|u|^p-1)^2$ (integrals with respect to
$dx$, i.e. `vol=|Ω|`), $\mu=200$, learning rate decaying from $10^{-3}$ to $10^{-5}$ (cosine) over 1500 epochs per
node, no early stopping, warm start. The curve is the loss on $4\times10^4$ fresh points normalized by
$(\int|u|^p)^{2(p-1)/p}$; around each minimum a local three-node scan with half the step is run, and the eigenvalue
is the frozen-network minimizer
$$E_{\rm PINN}=-\frac{\int\Delta_pu\,|u|^{p-2}u}{\int|u|^{2(p-1)}}.$$
No Weinstein-type certificate exists for $p\ne2$. `E_curve` is the vertex of the parabola through the coarse
minimum and its neighbours (less accurate). References: radial shooting and P1 finite elements on nodal subdomains
(`experiments/plap`).

Paper: disk $p=2.2$: $6.5337, 18.4069, 34.8043, 41.9615, 56.1166$; square $(0,2)^2$, $p=2.2$: $5.4902, 15.1440,
25.2285, 32.4615$; ball of $\mathbb R^3$, $p=2.5$: $14.1004$, $35.6283$; first eigenvalue of the disk for $p=3$
($9.801$, ref. $9.8315$) and $p=4$ ($14.895$, ref. $14.6816$; marginal). The regularized operator for $p<2$
(Table plap.reg) is in `experiments/plap` (`refs_reg.py`, jobs `REG` in `jobs.py`).

In [ ]:
res = run("plap_disk")

In [ ]:
res = run("plap_square")

In [ ]:
res = run("plap_ball3")

In [ ]:
res = run("plap_disk_p3")

In [ ]:
res = run("plap_disk_p4")

## Modifying an example or writing a new one

Any field of an `Example` can be changed with `dataclasses.replace` (penalty, epochs, window, start type, network
size, threshold, ...). A new problem only needs a sampler, a boundary factor $B$ (vanishing on $\partial\Omega$,
positive inside) and a potential. Below: the rectangle $(0,2)\times(0,1)$, whose Dirichlet eigenvalues are
$\pi^2(m^2/4+n^2)$.

In [ ]:
import torch, numpy as np

def sample_rect(n):                      # uniform points in (0,2)x(0,1)
    return torch.rand(n, 2) * torch.tensor([2.0, 1.0])

def boundary_rect(x):                    # B = x1 (2-x1) x2 (1-x2)
    return x[:, 0:1] * (2 - x[:, 0:1]) * x[:, 1:2] * (1 - x[:, 1:2])

RECT = Domain("rectangle (0,2)x(0,1)", 2, sample_rect, boundary_rect, vol=2.0, box=(0.0, 2.0))
ref = sorted((np.pi**2 * (m*m/4 + n*n), 1, f"({m},{n})") for m in range(1, 8) for n in range(1, 4))
rect = Example(name="rectangle", paper="(not in the paper)", section="-", domain=RECT,
               windows=[grid(8.0, 30.0, 1.0)], mu=200.0, n_epochs=2000, start="warm",
               threshold=2.0, reference=[t for t in ref if t[0] < 35])
if not PAPER_SETTINGS:
    rect = rect.quick()
res = run_example(rect, seed=SEED, out_dir=OUT, verbose=False)
from examples.core import print_summary
print_summary(rect, res)
display(Image(filename=res["figures"][0], width=720))

The JSON file written for each run contains the settings, one record per node (recorded residual, normalized loss,
mass, epochs, wall time) and, for every detection, the a posteriori quantities (`rho_res` and its standard error,
`r`, `eta`, Weinstein interval, Kato–Temple bound, relative difference to the reference).

In [ ]:
d = json.load(open(res["json"]))
print(json.dumps(d["detections"][:1], indent=1)[:1500])